In [25]:
import requests
import geopandas as gpd
import json
import pandas as pd
import os

In [14]:
# damage points (0), damage lines (1), damage polygons (2) 
layer_indices = [0, 1, 2]
base_url = "https://services.dat.noaa.gov/arcgis/rest/services/nws_damageassessmenttoolkit/DamageViewer/MapServer"

# storage for results
layer_data = {}

for num in layer_indices:
    print(f"Fetching layer {num}...")
    url = f"{base_url}/{num}/query"
    
    all_features = []
    offset = 0
    batch_size = 2000

    while True:
        params = {
            "f": "geojson",
            "where": "1=1",
            "outFields": "*",
            "resultRecordCount": batch_size,
            "resultOffset": offset
        }

        response = requests.get(url, params=params)
        data = response.json()

        features = data.get("features", [])
        if not features:
            break

        all_features.extend(features)
        offset += batch_size

    # store in geodataframe
    if all_features:
        gdf = gpd.GeoDataFrame.from_features(all_features)
        gdf.set_crs("EPSG:4326", inplace=True)
        layer_data[f"layer_{num}"] = gdf
        print(f"Layer {num}: {gdf.shape[0]} records")
    else:
        print(f"Layer {num}: No features found")

for name, df in layer_data.items():
    print(f"{name}: {df.shape}")


layer_names = {
    "layer_0": "damage_points",
    "layer_1": "damage_lines",
    "layer_2": "damage_polygons"
}

# Save to one GeoPackage with named layers
for key, df in layer_data.items():
    layer_label = layer_names.get(key, key)

    # Save to combined GeoPackage
    df.to_file("damage_data.gpkg", layer=layer_label, driver="GPKG")
    print(f"Saved layer '{layer_label}' to damage_data.gpkg")

    # Save to separate GPKG file
    df.to_file(f"{layer_label}.gpkg", layer=layer_label, driver="GPKG")
    print(f"Saved '{layer_label}' to its own file: {layer_label}.gpkg")


(10016, 21)

In [24]:
damage_points = gpd.read_file('damage_points.gpkg')
damage_points["stormdate"] = pd.to_datetime(damage_points["stormdate"], unit="ms")
damage_points["surveydate"] = pd.to_datetime(damage_points["surveydate"], unit="ms")
damage_points

,objectid,stormdate,surveydate,event_id,damage,damage_txt,dod_txt,efscale,damage_dir,windspeed,...,device_id,qc,dod,surveytype,globalid,edit_user,edit_time,comments,path_guid,geometry
0,25444,2010-04-30 00:00:00,2010-05-02 17:25:52,None,8.0,Small Retail Building [Fast Food Restaurants] ...,Uplift or collapse of entire roof structure,EF1,NE/45,0,...,31093a75,Y,6.0,None,{0FBCA08C-AD85-4C23-8F9E-D1C27D2B825F},None,NaN,west wall of a laundrymat collapsed and blew o...,{00000000-0000-0000-0000-000000000000},POINT (-94.0163 34.1237)
1,25445,2010-04-30 00:00:00,2010-05-02 17:34:26,None,23.0,Warehouse Building [Tilt-up Walls or Heavy-Tim...,Uplift of roof deck; significant loss of roof ...,EF1,NE/45,0,...,31093a75,Y,4.0,None,{3FEB48F9-F214-4795-A58E-BADE97EAB061},None,NaN,large section of roof peeled from a chicken house,{00000000-0000-0000-0000-000000000000},POINT (-94.0102 34.1259)
2,25467,2010-04-08 00:00:00,2010-04-09 00:00:00,None,28.0,Trees: Softwood (TS),Trees uprooted,EF0,UNKNOWN,85,...,None,Y,3.0,None,{A5526465-6D86-41E5-8849-8A16096DFEF3},None,NaN,Numerous pine trees down or topped.,{00000000-0000-0000-0000-000000000000},POINT (-84.59823 30.5302)
3,25468,2010-04-08 00:00:00,2010-04-09 11:32:08,None,27.0,Trees: Hardwood (TH),Trees uprooted,EF0,N/A,83,...,None,Y,3.0,None,{B22EECD4-AF02-4D5E-BF09-34246A3F9836},None,NaN,Large oak tree uprooted.,{00000000-0000-0000-0000-000000000000},POINT (-84.59432 30.53157)
4,25469,2010-04-08 00:00:00,2010-04-09 00:00:00,None,27.0,Trees: Hardwood (TH),Large branches broken (1-3 inch diameter),EF0,UNKNOWN,75,...,None,Y,2.0,None,{D7D4EC63-7301-4C22-AADA-8CB9F840D90D},None,NaN,Small oak tree snapped.,{00000000-0000-0000-0000-000000000000},POINT (-84.59514 30.53119)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195832,4476915,2025-04-28 00:45:00,2025-04-29 10:30:00,St. Francis Tornado,7.0,Masonry Apartment or Motel Building (MAM),Uplift of lightweight metal roof decking,TSTM/Wind,NNE/30,81,...,None,Y,3.0,None,{DC1578AB-4C2D-400D-AF3D-8DDE25C607B5},None,NaN,Section of damaged roof was on the SW side of ...,None,POINT (-100.90472 43.14854)
195833,4476916,2025-04-28 00:44:00,2025-04-29 10:30:00,St. Francis Tornado,9.0,"Small Professional Building [Doctor's Office, ...",Loss of roof covering (<20%),TSTM/Wind,NNE/30,65,...,None,Y,2.0,None,{E85783D3-7ACD-4A13-B768-BE68BBE56A6A},None,NaN,Section of roof damage was on the south side o...,None,POINT (-100.90607 43.1472)
195834,4476917,2025-04-28 00:40:00,2025-04-29 10:30:00,St. Francis Tornado,24.0,Electrical Transmission Lines (ETL),Threshold of visible damage,TSTM/Wind,NNE/30,70,...,None,Y,1.0,None,{089FAA1B-E0F8-4435-AEBE-DE6B3983035D},None,NaN,Wooden light post toppled over,None,POINT (-100.90466 43.14105)
195835,4476918,2025-04-28 00:40:00,2025-04-29 10:30:00,St. Francis Tornado,28.0,Trees: Softwood (TS),Trunks snapped,EF1,NNE/30,104,...,None,Y,4.0,None,{436C5C87-56C4-4681-8F2A-624AC82F9029},keith.sherburn_noaa,NaN,Numerous trees in this location uprooted.,None,POINT (-100.90453 43.14131)


In [30]:
"""Slow and downloads a low and higher res img for some reason. Need to figure out how to... not do that."""
# import os
# import requests
# 
# layer_id = 0
# out_dir = "dat_images_full"
# os.makedirs(out_dir, exist_ok=True)
# 
# object_ids = gdf["objectid"].dropna().astype(int).unique()
# 
# for obj_id in object_ids:
#     try:
#         # Step 1: Query for attachments
#         info_url = f"https://services.dat.noaa.gov/arcgis/rest/services/nws_damageassessmenttoolkit/DamageViewer/FeatureServer/{layer_id}/{obj_id}/attachments?f=json"
#         resp = requests.get(info_url)
#         data = resp.json()
#         attachments = data.get("attachmentInfos", [])
# 
#         for att in attachments:
#             att_id = att["id"]
#             filename = att["name"]
# 
#             # Step 2: Download actual image
#             download_url = f"https://services.dat.noaa.gov/arcgis/rest/services/nws_damageassessmenttoolkit/DamageViewer/FeatureServer/{layer_id}/{obj_id}/attachments/{att_id}"
#             out_path = os.path.join(out_dir, f"{obj_id}_{att_id}_{filename}")
# 
#             img = requests.get(download_url)
#             if img.status_code == 200:
#                 with open(out_path, "wb") as f:
#                     f.write(img.content)
#             else:
#                 print(f"Failed: {download_url}")
#     except Exception as e:
#         print(f"Error on objectid {obj_id}: {e}")


ERROR! Session/line number was not unique in database. History logging moved to new session 600



KeyboardInterrupt



In [41]:
valid_ef = ["EF0", "EF1", "EF2", "EF3", "EF4", "EF5", "EFU", "UNKNOWN", "N/A"]
tornado_points_df = damage_points[damage_points.efscale.isin(valid_ef)]

In [47]:
tornado_points_df[tornado_points_df.efscale == "UNKNOWN"].comments

3269      Cactus Drilling Rig #117 hit by tornado.  Seve...
3337      .Structure gone, no debris in the area, do not...
3354      Via CAP overflight photo.  Unsure of degree of...
3356                              Via CAP overflight photo.
3387                              Via CAP overflight photo.
                                ...                        
171269    Brief spinup occurred in open field and lasted...
171270                        End of tornado path per video
171596    The tornado continued to intensify as it quick...
171597    The tornado likely continued to travel slowly ...
171599    This tornado was reported to have dissipated a...
Name: comments, Length: 1623, dtype: object

In [44]:
tornado_points_df[tornado_points_df["image"].notnull()].groupby("efscale").size().reset_index(name="image_count")

,efscale,image_count
0,EF0,45403
1,EF1,42175
2,EF2,8958
3,EF3,2214
4,EF4,825
5,EF5,88
6,EFU,448
7,N/A,2408
8,UNKNOWN,1476


Get images for higher end tornadoes. Try to retrieve more at your own risk... (many many images and each has to be 2 separate queries)

In [48]:
# import pandas as pd
# import requests
# from concurrent.futures import ThreadPoolExecutor, as_completed
# 
# # Filter to EF4 and EF5 first
# ef4_5_df = tornado_points_df[tornado_points_df["efscale"].isin(["EF4", "EF5"])]
# object_ids = ef4_5_df["objectid"].dropna().astype(int).unique()
# 
# def fetch_attachments(obj_id):
#     try:
#         url = f"https://services.dat.noaa.gov/arcgis/rest/services/nws_damageassessmenttoolkit/DamageViewer/FeatureServer/0/{obj_id}/attachments?f=json"
#         resp = requests.get(url, timeout=10).json()
#         return [
#             {
#                 "objectid": obj_id,
#                 "attachment_id": att["id"],
#                 "name": att["name"],
#                 "size": att.get("size")
#             }
#             for att in resp.get("attachmentInfos", [])
#         ]
#     except Exception as e:
#         return [{"objectid": obj_id, "error": str(e)}]
# 
# # Run with up to 8 concurrent threads (adjust as needed)
# attachment_records = []
# with ThreadPoolExecutor(max_workers=8) as executor:
#     futures = [executor.submit(fetch_attachments, oid) for oid in object_ids]
#     for future in as_completed(futures):
#         result = future.result()
#         attachment_records.extend(result)
# 
# import os
# 
# valid_records = [r for r in attachment_records if "error" not in r]
# imgurl_df = pd.DataFrame(valid_records)
# 
# imgurl_df = imgurl_df.merge(
#     ef4_5_df[["objectid", "efscale"]],
#     on="objectid",
#     how="left"
# )
# 
# imgurl_df["image_url"] = imgurl_df.apply(
#     lambda row: f"https://services.dat.noaa.gov/arcgis/rest/services/nws_damageassessmenttoolkit/DamageViewer/FeatureServer/0/{row.objectid}/attachments/{row.attachment_id}",
#     axis=1
# )
# 
# os.makedirs("EF4_EF5_Images", exist_ok=True)
# imgurl_df["download_status"] = "pending"
# 
# # Download with status tracking
# for i, row in imgurl_df.iterrows():
#     try:
#         url = row["image_url"]
#         filename = f'{row["objectid"]}_{row["attachment_id"]}_{row["name"]}'
#         path = os.path.join("EF4_EF5_Images", filename)
# 
#         r = requests.get(url, timeout=10)
#         if r.status_code == 200:
#             with open(path, "wb") as f:
#                 f.write(r.content)
#             imgurl_df.at[i, "download_status"] = "success"
#         else:
#             imgurl_df.at[i, "download_status"] = f"HTTP {r.status_code}"
#     except Exception as e:
#         imgurl_df.at[i, "download_status"] = f"error: {str(e)}"
# 
# # Save log-enhanced CSV
# imgurl_df.to_csv("ef4_ef5_image_log.csv", index=False)


KeyboardInterrupt: 

In [49]:
damage_points

,objectid,stormdate,surveydate,event_id,damage,damage_txt,dod_txt,efscale,damage_dir,windspeed,...,device_id,qc,dod,surveytype,globalid,edit_user,edit_time,comments,path_guid,geometry
0,25444,2010-04-30 00:00:00,2010-05-02 17:25:52,None,8.0,Small Retail Building [Fast Food Restaurants] ...,Uplift or collapse of entire roof structure,EF1,NE/45,0,...,31093a75,Y,6.0,None,{0FBCA08C-AD85-4C23-8F9E-D1C27D2B825F},None,NaN,west wall of a laundrymat collapsed and blew o...,{00000000-0000-0000-0000-000000000000},POINT (-94.0163 34.1237)
1,25445,2010-04-30 00:00:00,2010-05-02 17:34:26,None,23.0,Warehouse Building [Tilt-up Walls or Heavy-Tim...,Uplift of roof deck; significant loss of roof ...,EF1,NE/45,0,...,31093a75,Y,4.0,None,{3FEB48F9-F214-4795-A58E-BADE97EAB061},None,NaN,large section of roof peeled from a chicken house,{00000000-0000-0000-0000-000000000000},POINT (-94.0102 34.1259)
2,25467,2010-04-08 00:00:00,2010-04-09 00:00:00,None,28.0,Trees: Softwood (TS),Trees uprooted,EF0,UNKNOWN,85,...,None,Y,3.0,None,{A5526465-6D86-41E5-8849-8A16096DFEF3},None,NaN,Numerous pine trees down or topped.,{00000000-0000-0000-0000-000000000000},POINT (-84.59823 30.5302)
3,25468,2010-04-08 00:00:00,2010-04-09 11:32:08,None,27.0,Trees: Hardwood (TH),Trees uprooted,EF0,N/A,83,...,None,Y,3.0,None,{B22EECD4-AF02-4D5E-BF09-34246A3F9836},None,NaN,Large oak tree uprooted.,{00000000-0000-0000-0000-000000000000},POINT (-84.59432 30.53157)
4,25469,2010-04-08 00:00:00,2010-04-09 00:00:00,None,27.0,Trees: Hardwood (TH),Large branches broken (1-3 inch diameter),EF0,UNKNOWN,75,...,None,Y,2.0,None,{D7D4EC63-7301-4C22-AADA-8CB9F840D90D},None,NaN,Small oak tree snapped.,{00000000-0000-0000-0000-000000000000},POINT (-84.59514 30.53119)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195832,4476915,2025-04-28 00:45:00,2025-04-29 10:30:00,St. Francis Tornado,7.0,Masonry Apartment or Motel Building (MAM),Uplift of lightweight metal roof decking,TSTM/Wind,NNE/30,81,...,None,Y,3.0,None,{DC1578AB-4C2D-400D-AF3D-8DDE25C607B5},None,NaN,Section of damaged roof was on the SW side of ...,None,POINT (-100.90472 43.14854)
195833,4476916,2025-04-28 00:44:00,2025-04-29 10:30:00,St. Francis Tornado,9.0,"Small Professional Building [Doctor's Office, ...",Loss of roof covering (<20%),TSTM/Wind,NNE/30,65,...,None,Y,2.0,None,{E85783D3-7ACD-4A13-B768-BE68BBE56A6A},None,NaN,Section of roof damage was on the south side o...,None,POINT (-100.90607 43.1472)
195834,4476917,2025-04-28 00:40:00,2025-04-29 10:30:00,St. Francis Tornado,24.0,Electrical Transmission Lines (ETL),Threshold of visible damage,TSTM/Wind,NNE/30,70,...,None,Y,1.0,None,{089FAA1B-E0F8-4435-AEBE-DE6B3983035D},None,NaN,Wooden light post toppled over,None,POINT (-100.90466 43.14105)
195835,4476918,2025-04-28 00:40:00,2025-04-29 10:30:00,St. Francis Tornado,28.0,Trees: Softwood (TS),Trunks snapped,EF1,NNE/30,104,...,None,Y,4.0,None,{436C5C87-56C4-4681-8F2A-624AC82F9029},keith.sherburn_noaa,NaN,Numerous trees in this location uprooted.,None,POINT (-100.90453 43.14131)


Damage Points:

objectid : 
event_id :
path_guid :
globalid :
device_id :

stormdate :
surveydate :
edit_time :

damage : 
damage_txt :
dod_txt :
efscale :
dod :
windspeed :
qc :

injuries :
deaths :

lat :
lon :
gps_horiz_accuracy :
geometry :

office :
surveytype :
edit_user :
image :

comments :


In [54]:
damage_points.columns

Index(['objectid', 'stormdate', 'surveydate', 'event_id', 'damage',
       'damage_txt', 'dod_txt', 'efscale', 'damage_dir', 'windspeed',
       'injuries', 'deaths', 'lat', 'lon', 'office', 'image',
       'gps_horiz_accuracy', 'device_id', 'qc', 'dod', 'surveytype',
       'globalid', 'edit_user', 'edit_time', 'comments', 'path_guid',
       'geometry'],
      dtype='object')

In [55]:
damage_lines = gpd.read_file('damage/damage_lines.gpkg')

In [56]:
damage_lines.columns

Index(['objectid', 'event_id', 'stormdate', 'starttime', 'endtime', 'startlat',
       'startlon', 'endlat', 'endlon', 'length', 'width', 'injuries',
       'fatalities', 'efscale', 'efnum', 'qc', 'maxwind', 'globalid',
       'cropdamage', 'propdamage', 'edit_user', 'edit_time', 'created_user',
       'created_date', 'last_edited_user', 'last_edited_date', 'comments',
       'st_length(shape)', 'wfo', 'geometry'],
      dtype='object')

In [57]:
damage_lines

,objectid,event_id,stormdate,starttime,endtime,startlat,startlon,endlat,endlon,length,...,edit_user,edit_time,created_user,created_date,last_edited_user,last_edited_date,comments,st_length(shape),wfo,geometry
0,2555,None,1321444320000,1321444320000,1321446900000,32.5370,-85.6566,32.6970,-85.1173,33.8100,...,None,None,None,NaN,DAT,1530238731000,A TORNADO TOUCHED DOWN IN NORTHEAST MACON COUN...,0.568802,None,"LINESTRING (-85.65659 32.53697, -85.64985 32.5..."
1,2559,None,1324560600000,1324560600000,1324647180000,32.4231,-86.9206,32.4305,-86.8844,2.1900,...,None,None,None,NaN,DAT,1530238731000,"Dallas County EF-0 Tornado Path - December 22,...",0.037152,None,"LINESTRING (-86.92058 32.42309, -86.91741 32.4..."
2,2562,None,1324561320000,1324561320000,1324561500000,33.1828,-86.6070,33.2027,-86.5860,1.9400,...,None,None,None,NaN,DAT,1530238731000,"Columbiana EF-0 Tornado - December 22, 2011.",0.030627,None,"LINESTRING (-86.60701 33.18281, -86.60428 33.1..."
3,2563,None,1327286520000,1327286520000,1327286580000,33.3257,-87.6579,33.3305,-87.6526,0.4500,...,None,None,None,NaN,DAT,1530238731000,"Koffman EF-2 Tornado - January 23, 2012.",0.007155,None,"LINESTRING (-87.65791 33.32572, -87.65578 33.3..."
4,2564,None,1327287600000,1327287600000,1327287660000,33.3961,-87.4440,33.4038,-87.4395,0.6400,...,None,None,None,NaN,DAT,1530238731000,Watermelon Road EF-2 Tornado Damage Path - Jan...,0.009740,None,"LINESTRING (-87.44401 33.39611, -87.4424 33.39..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10787,2393573,,1745192880000,1745192880000,1745193120000,39.7910,-91.3110,39.8194,-91.2424,4.1850,...,None,None,dat_editor,1.745348e+12,dat_editor,1745352423000,,0.074817,LSX,"LINESTRING (-91.31102 39.79096, -91.259 39.808..."
10788,2393971,West of Cundiff,1745105340000,1745105340000,1745105640000,33.3046,-98.0529,33.3212,-98.0146,2.5105,...,None,None,dat_editor,1.745280e+12,dat_editor,1745819003000,The third tornado in Jack County on the evenin...,0.041956,FWD,"LINESTRING (-98.05286 33.3046, -98.04437 33.30..."
10789,2394371,Near Adell,1745121660000,1745121660000,1745122200000,32.8536,-97.9391,32.9098,-97.9049,4.3596,...,None,None,dat_editor,1.745339e+12,dat_editor,1746006206000,An EF-1 tornado with maximum estimated winds o...,0.065899,FWD,"LINESTRING (-97.93905 32.85359, -97.93731 32.8..."
10790,2394772,SE Faribault EF1,1745880720000,1745880720000,1745880900000,44.2542,-93.1190,44.2607,-93.0538,3.2747,...,None,None,dat_editor,1.746018e+12,dat_editor,1746031621000,Preliminary EF1 Tornado Rice County,0.065607,MPX,"LINESTRING (-93.11902 44.25424, -93.11083 44.2..."


In [58]:
damage_lines["stormdate"] = pd.to_datetime(damage_lines["stormdate"], unit="ms")
damage_lines["starttime"] = pd.to_datetime(damage_lines["starttime"], unit="ms")
damage_lines["endtime"] = pd.to_datetime(damage_lines["endtime"], unit="ms")
damage_lines["last_edited_date"] = pd.to_datetime(damage_lines["last_edited_date"], unit="ms")

damage_lines

,objectid,event_id,stormdate,starttime,endtime,startlat,startlon,endlat,endlon,length,...,edit_user,edit_time,created_user,created_date,last_edited_user,last_edited_date,comments,st_length(shape),wfo,geometry
0,2555,None,2011-11-16 11:52:00,2011-11-16 11:52:00,2011-11-16 12:35:00,32.5370,-85.6566,32.6970,-85.1173,33.8100,...,None,None,None,NaN,DAT,2018-06-29 02:18:51,A TORNADO TOUCHED DOWN IN NORTHEAST MACON COUN...,0.568802,None,"LINESTRING (-85.65659 32.53697, -85.64985 32.5..."
1,2559,None,2011-12-22 13:30:00,2011-12-22 13:30:00,2011-12-23 13:33:00,32.4231,-86.9206,32.4305,-86.8844,2.1900,...,None,None,None,NaN,DAT,2018-06-29 02:18:51,"Dallas County EF-0 Tornado Path - December 22,...",0.037152,None,"LINESTRING (-86.92058 32.42309, -86.91741 32.4..."
2,2562,None,2011-12-22 13:42:00,2011-12-22 13:42:00,2011-12-22 13:45:00,33.1828,-86.6070,33.2027,-86.5860,1.9400,...,None,None,None,NaN,DAT,2018-06-29 02:18:51,"Columbiana EF-0 Tornado - December 22, 2011.",0.030627,None,"LINESTRING (-86.60701 33.18281, -86.60428 33.1..."
3,2563,None,2012-01-23 02:42:00,2012-01-23 02:42:00,2012-01-23 02:43:00,33.3257,-87.6579,33.3305,-87.6526,0.4500,...,None,None,None,NaN,DAT,2018-06-29 02:18:51,"Koffman EF-2 Tornado - January 23, 2012.",0.007155,None,"LINESTRING (-87.65791 33.32572, -87.65578 33.3..."
4,2564,None,2012-01-23 03:00:00,2012-01-23 03:00:00,2012-01-23 03:01:00,33.3961,-87.4440,33.4038,-87.4395,0.6400,...,None,None,None,NaN,DAT,2018-06-29 02:18:51,Watermelon Road EF-2 Tornado Damage Path - Jan...,0.009740,None,"LINESTRING (-87.44401 33.39611, -87.4424 33.39..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10787,2393573,,2025-04-20 23:48:00,2025-04-20 23:48:00,2025-04-20 23:52:00,39.7910,-91.3110,39.8194,-91.2424,4.1850,...,None,None,dat_editor,1.745348e+12,dat_editor,2025-04-22 20:07:03,,0.074817,LSX,"LINESTRING (-91.31102 39.79096, -91.259 39.808..."
10788,2393971,West of Cundiff,2025-04-19 23:29:00,2025-04-19 23:29:00,2025-04-19 23:34:00,33.3046,-98.0529,33.3212,-98.0146,2.5105,...,None,None,dat_editor,1.745280e+12,dat_editor,2025-04-28 05:43:23,The third tornado in Jack County on the evenin...,0.041956,FWD,"LINESTRING (-98.05286 33.3046, -98.04437 33.30..."
10789,2394371,Near Adell,2025-04-20 04:01:00,2025-04-20 04:01:00,2025-04-20 04:10:00,32.8536,-97.9391,32.9098,-97.9049,4.3596,...,None,None,dat_editor,1.745339e+12,dat_editor,2025-04-30 09:43:26,An EF-1 tornado with maximum estimated winds o...,0.065899,FWD,"LINESTRING (-97.93905 32.85359, -97.93731 32.8..."
10790,2394772,SE Faribault EF1,2025-04-28 22:52:00,2025-04-28 22:52:00,2025-04-28 22:55:00,44.2542,-93.1190,44.2607,-93.0538,3.2747,...,None,None,dat_editor,1.746018e+12,dat_editor,2025-04-30 16:47:01,Preliminary EF1 Tornado Rice County,0.065607,MPX,"LINESTRING (-93.11902 44.25424, -93.11083 44.2..."


In [66]:
damage_polygons = gpd.read_file('damage/damage_polygons.gpkg')

In [70]:
damage_polygons["stormdate"] = pd.to_datetime(damage_polygons["stormdate"], unit="ms")
damage_polygons["created_date"] = pd.to_datetime(damage_polygons["created_date"], unit="ms")
damage_polygons["last_edited_date"] = pd.to_datetime(damage_polygons["last_edited_date"], unit="ms")

In [71]:
damage_polygons

,objectid,efscale,qc,globalid,event_id,stormdate,length,width,injuries,fatalities,...,edit_time,created_user,created_date,last_edited_user,last_edited_date,comments,path_guid,st_area(shape),st_perimeter(shape),geometry
0,5480,EF5,Y,{BFE42AFE-7B47-45E6-8495-8073C70C61CF},None,2011-04-27 15:05:00,132.0,2200.0,-99.0,-99.0,...,None,None,NaT,DAT,2018-06-29 02:21:46,EF5 Oak Grove damage. Hackleburg-Phil Campbel...,{00000000-0000-0000-0000-000000000000},4.246624e-04,0.272121,"MULTIPOLYGON (((-87.58553 34.40756, -87.58502 ..."
1,5481,EF0,Y,{894B6053-6738-4226-BE23-8F3A6EBFB9D6},None,NaT,-99.0,-99.0,-99.0,-99.0,...,None,None,NaT,DAT,2018-06-29 02:21:46,NULL,{00000000-0000-0000-0000-000000000000},9.695575e-05,0.080354,"MULTIPOLYGON (((-95.70306 32.43086, -95.70032 ..."
2,5482,EF4,Y,{BBEB80CE-7740-468D-9D17-25420E612185},None,2011-04-27 15:05:00,132.0,2200.0,-99.0,-99.0,...,None,None,NaT,DAT,2018-06-29 02:21:46,Phil Campbell to Oak Grove EF4,{00000000-0000-0000-0000-000000000000},1.296908e-03,0.693048,"MULTIPOLYGON (((-87.46457 34.487, -87.475 34.4..."
3,5484,EF1,Y,{4FDA2C31-4A9B-4AAD-B724-8CD58C6CBC8A},None,NaT,-99.0,-99.0,-99.0,-99.0,...,None,None,NaT,DAT,2018-06-29 02:21:46,NULL,{00000000-0000-0000-0000-000000000000},1.665674e-04,0.066724,"MULTIPOLYGON (((-95.64802 32.45914, -95.64888 ..."
4,5485,EF1,Y,{CB30B408-82AF-4CB5-8F05-A6C3BAF17714},None,NaT,-99.0,-99.0,-99.0,-99.0,...,None,None,NaT,DAT,2018-06-29 02:21:46,NULL,{00000000-0000-0000-0000-000000000000},4.285275e-05,0.030208,"MULTIPOLYGON (((-95.59515 32.49013, -95.59944 ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10011,2042617,EF2,Y,{463289AB-8A39-4DAF-A996-1A9501194F98},"Lake City, AR",2025-04-02 23:33:00,0.0,0.0,0.0,0.0,...,None,dat_editor,2025-04-15 15:02:40,dat_editor,2025-04-23 20:24:27,,None,2.259568e-05,0.032901,"MULTIPOLYGON (((-90.54559 35.73785, -90.54494 ..."
10012,2043018,EF1,Y,{506D2154-D95B-4994-8ACD-F76FE17EEE6E},Slayden/Grand Junction (MS/TN),2025-04-03 06:14:00,0.0,0.0,0.0,0.0,...,None,dat_editor,2025-04-15 19:36:12,dat_editor,2025-04-15 19:56:08,,None,4.034645e-03,1.346074,"MULTIPOLYGON (((-89.50487 34.891, -89.50092 34..."
10013,2043019,EF0,Y,{C604EE63-C49F-4EF8-83FD-DA3551B95588},,2025-04-04 21:24:00,0.0,0.0,0.0,0.0,...,None,dat_editor,2025-04-22 19:34:39,dat_editor,2025-04-22 20:23:52,,None,4.139851e-03,0.874252,"MULTIPOLYGON (((-95.33347 32.55557, -95.33674 ..."
10014,2043021,EF1,Y,{1E2A125D-07A5-47E6-8BE0-A1E1D19485FA},Red River County TX,2025-04-04 20:27:00,0.0,0.0,0.0,0.0,...,None,dat_editor,2025-04-22 21:01:50,dat_editor,2025-04-22 21:15:46,,None,1.512683e-07,0.001599,"MULTIPOLYGON (((-95.03533 33.61335, -95.03496 ..."


In [73]:
import nbformat

def repair_notebook(path_in, path_out=None):
    nb = nbformat.read(path_in, as_version=4)

    for i, cell in enumerate(nb.cells):
        cell.setdefault("metadata", {})

        if cell.cell_type == "code":
            cell.setdefault("execution_count", None)
            cell.setdefault("outputs", [])
            cell.setdefault("source", "")

        elif cell.cell_type == "markdown":
            cell.setdefault("source", "")

    nb.metadata.setdefault("language_info", {})
    nb.metadata.setdefault("kernelspec", {})
    
    path_out = path_out or path_in
    nbformat.write(nb, path_out)

repair_notebook("damage_assessment.ipynb")
